# Yiddish video → English subtitles pipeline

Event-driven pipeline on AWS:

1. Upload a video to S3 → an **ffmpeg Lambda** extracts a 16 kHz mono WAV.
2. A **transcribe Lambda** sends the WAV to the **word-level Whisper** async endpoint
   (`whisper-yi-word`), which writes word-timestamped output to S3.
3. A **translate Lambda** groups words into sentence cues, translates them in context
   with Claude on Bedrock, and writes `.en.json` + `.en.srt` subtitles to S3.

A single GPU endpoint (`whisper-yi-word`) serves transcription; word-level timestamps are
a superset of chunk-level, so the older `whisper-yi-model` endpoint is no longer needed
(see the decommission cell at the end).

In [ ]:
!pip install -q "sagemaker<3.0.0"

In [1]:
# --- config + clients (run first) ---
import os, io, json, time, zipfile, tarfile, urllib.request
import boto3, sagemaker
from urllib.parse import urlparse
from botocore.exceptions import ClientError
from sagemaker.huggingface import HuggingFaceModel
from sagemaker.async_inference import AsyncInferenceConfig

region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]
s3 = boto3.client("s3"); iam = boto3.client("iam")
lam = boto3.client("lambda"); sm = boto3.client("sagemaker")

sess = sagemaker.Session()
bucket = sess.default_bucket()          # SageMaker default bucket (async output lands here)
try:
    role = sagemaker.get_execution_role()
except Exception:
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

model_data = "s3://sagemaker-us-east-1-346399954218/whisper/ivrit-ai-yi-whisper-large-v3/model.tar.gz"

# endpoints / prefixes
WORD_ENDPOINT = "whisper-yi-word"
WORD_OUTPUT_PREFIX = "whisper/word-output/"     # async output of WORD_ENDPOINT (in `bucket`)

PIPELINE_BUCKET = f"whisper-video-pipeline-{account_id}-{region}"
VIDEO_PREFIX = "videos/"
AUDIO_PREFIX = "audio/"
EN_PREFIX = "transcripts-en/"
LAYER_KEY = "layers/ffmpeg-static-layer.zip"

# lambdas / roles
FFMPEG_FN, FFMPEG_ROLE = "whisper-ffmpeg-extract-audio", "whisper-ffmpeg-extract-role"
TRANSCRIBE_FN, TRANSCRIBE_ROLE = "whisper-invoke-transcribe", "whisper-invoke-transcribe-role"
TRANSLATE_FN, TRANSLATE_ROLE = "whisper-translate-subtitles", "whisper-translate-subtitles-role"
FFMPEG_LAYER = "ffmpeg-static-layer"

# bedrock
MODEL_ID = "us.anthropic.claude-opus-4-8"
BEDROCK_REGION = region


def zip_lambda(src):
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("lambda_function.py", src)
    return buf.getvalue()


print("account:", account_id, "| region:", region)
print("pipeline bucket:", PIPELINE_BUCKET)
print("endpoint:", WORD_ENDPOINT)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
account: 346399954218 | region: us-east-1
pipeline bucket: whisper-video-pipeline-346399954218-us-east-1
endpoint: whisper-yi-word


In [ ]:
# --- dedicated least-privilege execution role for the Whisper endpoint ---
# Replaces the broad SageMaker notebook role so the endpoint can do ONLY what it needs:
# pull the DLC image, read its model artifact + the audio input, write async output, log.
ENDPOINT_ROLE = "whisper-yi-word-endpoint-role"
DLC_ACCOUNT = "763104351884"   # AWS-managed Hugging Face DLC registry

ep_trust = {"Version": "2012-10-17", "Statement": [{
    "Effect": "Allow", "Principal": {"Service": "sagemaker.amazonaws.com"},
    "Action": "sts:AssumeRole",
    "Condition": {"StringEquals": {"aws:SourceAccount": account_id}}}]}
try:
    ENDPOINT_ROLE_ARN = iam.create_role(
        RoleName=ENDPOINT_ROLE, AssumeRolePolicyDocument=json.dumps(ep_trust),
        Description="Least-privilege execution role for whisper-yi-word")["Role"]["Arn"]
except iam.exceptions.EntityAlreadyExistsException:
    iam.update_assume_role_policy(RoleName=ENDPOINT_ROLE, PolicyDocument=json.dumps(ep_trust))
    ENDPOINT_ROLE_ARN = iam.get_role(RoleName=ENDPOINT_ROLE)["Role"]["Arn"]

ep_policy = {"Version": "2012-10-17", "Statement": [
    {"Sid": "PullDLCImageLayers", "Effect": "Allow",
     "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer",
                "ecr:BatchCheckLayerAvailability"],
     "Resource": f"arn:aws:ecr:{region}:{DLC_ACCOUNT}:repository/huggingface-pytorch-inference"},
    {"Sid": "EcrAuthToken", "Effect": "Allow",
     "Action": ["ecr:GetAuthorizationToken"], "Resource": "*"},
    {"Sid": "ReadModelArtifact", "Effect": "Allow", "Action": ["s3:GetObject"],
     "Resource": f"arn:aws:s3:::{bucket}/*model.tar.gz"},
    {"Sid": "ReadAudioInput", "Effect": "Allow", "Action": ["s3:GetObject"],
     "Resource": f"arn:aws:s3:::{PIPELINE_BUCKET}/{AUDIO_PREFIX}*"},
    {"Sid": "WriteAsyncOutput", "Effect": "Allow", "Action": ["s3:PutObject"],
     "Resource": [f"arn:aws:s3:::{bucket}/whisper/word-output/*",
                  f"arn:aws:s3:::{bucket}/whisper/word-failure/*"]},
    {"Sid": "Logs", "Effect": "Allow",
     "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
     "Resource": f"arn:aws:logs:{region}:{account_id}:log-group:/aws/sagemaker/Endpoints/{WORD_ENDPOINT}*"},
    {"Sid": "Metrics", "Effect": "Allow", "Action": ["cloudwatch:PutMetricData"],
     "Resource": "*",
     "Condition": {"StringEquals": {"cloudwatch:namespace": ["/aws/sagemaker/Endpoints", "AWS/SageMaker"]}}},
]}
iam.put_role_policy(RoleName=ENDPOINT_ROLE, PolicyName="inline",
                    PolicyDocument=json.dumps(ep_policy))
time.sleep(12)
print("Endpoint execution role:", ENDPOINT_ROLE_ARN)

## 1. Word-level Whisper endpoint

### Convert the model to CTranslate2 (one-time)

faster-whisper runs on a CTranslate2 build of the model. This cell downloads the `ivrit-ai` checkpoint, converts it (fp16), packages it, and uploads it to S3 as `CT2_MODEL_DATA`. Run once; rerun only to rebuild the model artifact.

In [ ]:
# --- one-time: convert ivrit-ai/yi-whisper-large-v3 to CTranslate2 ---
import subprocess, sys, tarfile

HF_MODEL_ID = "ivrit-ai/yi-whisper-large-v3"
CT2_DIR = "/tmp/ct2-yi-whisper"
CT2_PREFIX = "whisper/ct2-yi-whisper-large-v3"

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "ctranslate2==4.5.0", "transformers", "huggingface_hub", "torch"])

# Convert: writes a CT2 model dir incl. tokenizer.json + preprocessor_config.json.
subprocess.check_call([
    "ct2-transformers-converter", "--model", HF_MODEL_ID, "--output_dir", CT2_DIR,
    "--copy_files", "tokenizer.json", "preprocessor_config.json",
    "--quantization", "float16", "--force",
])

# Tar with files at the archive root, then upload -> CT2_MODEL_DATA.
tar_path = "/tmp/ct2_model.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    for name in os.listdir(CT2_DIR):
        tar.add(os.path.join(CT2_DIR, name), arcname=name)

CT2_MODEL_DATA = sess.upload_data(path=tar_path, bucket=bucket, key_prefix=CT2_PREFIX)
print("CT2 model uploaded:", CT2_MODEL_DATA)

In [ ]:
# --- deploy the faster-whisper (CTranslate2) word-level endpoint ---
for fn, kw in [(sm.delete_endpoint, {"EndpointName": WORD_ENDPOINT}),
               (sm.delete_endpoint_config, {"EndpointConfigName": WORD_ENDPOINT}),
               (sm.delete_model, {"ModelName": WORD_ENDPOINT})]:
    try:
        fn(**kw)
    except Exception:
        pass

word_model = HuggingFaceModel(
    model_data=CT2_MODEL_DATA,
    transformers_version="4.49.0",
    pytorch_version="2.6.0",
    py_version="py312",
    role=ENDPOINT_ROLE_ARN,
    entry_point="inference.py",
    source_dir="code_whisper",
    name=WORD_ENDPOINT,
    env={
        "compute_type": "float16",     # GPU; falls back to int8 automatically on CPU
        "language": "yi",              # Yiddish (skip language detection)
        "beam_size": "5",
        "vad_filter": "true",          # Silero VAD -> cleaner long-form timing
        "MMS_MAX_REQUEST_SIZE": "2000000000",
        "MMS_MAX_RESPONSE_SIZE": "2000000000",
        "MMS_DEFAULT_RESPONSE_TIMEOUT": "3600",
        "TS_DEFAULT_RESPONSE_TIMEOUT": "3600",
    },
)

word_predictor = word_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name=WORD_ENDPOINT,
    async_inference_config=AsyncInferenceConfig(
        output_path=f"s3://{bucket}/whisper/word-output",
        failure_path=f"s3://{bucket}/whisper/word-failure",
    ),
)
print("Deployed faster-whisper endpoint:", WORD_ENDPOINT)

## 2. Ingestion: S3 bucket + ffmpeg Lambda + transcribe Lambda

In [20]:
# --- create + harden the pipeline bucket ---
try:
    if region == "us-east-1":
        s3.create_bucket(Bucket=PIPELINE_BUCKET)
    else:
        s3.create_bucket(Bucket=PIPELINE_BUCKET,
                         CreateBucketConfiguration={"LocationConstraint": region})
    print("Created", PIPELINE_BUCKET)
except s3.exceptions.BucketAlreadyOwnedByYou:
    print("Bucket already owned by you")

s3.put_public_access_block(
    Bucket=PIPELINE_BUCKET,
    PublicAccessBlockConfiguration={"BlockPublicAcls": True, "IgnorePublicAcls": True,
                                    "BlockPublicPolicy": True, "RestrictPublicBuckets": True})

# Explicit default encryption at rest (SSE-S3 + bucket key).
s3.put_bucket_encryption(
    Bucket=PIPELINE_BUCKET,
    ServerSideEncryptionConfiguration={"Rules": [
        {"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"},
         "BucketKeyEnabled": True}]})

# Explicit default encryption at rest (SSE-S3 + bucket key).
s3.put_bucket_encryption(
    Bucket=PIPELINE_BUCKET,
    ServerSideEncryptionConfiguration={"Rules": [
        {"ApplyServerSideEncryptionByDefault": {"SSEAlgorithm": "AES256"},
         "BucketKeyEnabled": True}]})

s3.put_bucket_lifecycle_configuration(
    Bucket=PIPELINE_BUCKET,
    LifecycleConfiguration={"Rules": [
        {"ID": "expire-videos", "Filter": {"Prefix": VIDEO_PREFIX}, "Status": "Enabled",
         "Expiration": {"Days": 7}},
        {"ID": "expire-audio", "Filter": {"Prefix": AUDIO_PREFIX}, "Status": "Enabled",
         "Expiration": {"Days": 7}},
        {"ID": "abort-mpu", "Filter": {"Prefix": ""}, "Status": "Enabled",
         "AbortIncompleteMultipartUpload": {"DaysAfterInitiation": 3}}]})

# Let the Whisper endpoint execution role read the extracted WAV; enforce TLS.
bucket_policy = {"Version": "2012-10-17", "Statement": [
    {"Sid": "AllowSageMakerRoleReadAudio", "Effect": "Allow",
     "Principal": {"AWS": ENDPOINT_ROLE_ARN}, "Action": ["s3:GetObject"],
     "Resource": f"arn:aws:s3:::{PIPELINE_BUCKET}/{AUDIO_PREFIX}*"},
    {"Sid": "DenyInsecureTransport", "Effect": "Deny", "Principal": "*", "Action": "s3:*",
     "Resource": [f"arn:aws:s3:::{PIPELINE_BUCKET}", f"arn:aws:s3:::{PIPELINE_BUCKET}/*"],
     "Condition": {"Bool": {"aws:SecureTransport": "false"}}}]}
s3.put_bucket_policy(Bucket=PIPELINE_BUCKET, Policy=json.dumps(bucket_policy))
print("Bucket hardened")

Created whisper-video-pipeline-346399954218-us-east-1
Bucket hardened


In [4]:
# --- build + publish the ffmpeg Lambda layer (binary at /opt/bin/ffmpeg) ---
work = "/tmp/ffmpeg_layer"; os.makedirs(work, exist_ok=True)
tar_xz = "/tmp/ffmpeg.tar.xz"
urllib.request.urlretrieve(
    "https://johnvansickle.com/ffmpeg/releases/ffmpeg-release-amd64-static.tar.xz", tar_xz)
bin_path = os.path.join(work, "ffmpeg")
with tarfile.open(tar_xz) as t:
    m = next(x for x in t.getmembers() if x.name.endswith("/ffmpeg"))
    with t.extractfile(m) as src, open(bin_path, "wb") as dst:
        dst.write(src.read())

layer_zip = "/tmp/ffmpeg_layer.zip"
zi = zipfile.ZipInfo("bin/ffmpeg"); zi.external_attr = 0o755 << 16
with open(bin_path, "rb") as f, zipfile.ZipFile(layer_zip, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr(zi, f.read())

s3.upload_file(layer_zip, PIPELINE_BUCKET, LAYER_KEY)
ffmpeg_layer_arn = lam.publish_layer_version(
    LayerName=FFMPEG_LAYER, Description="Static ffmpeg at /opt/bin/ffmpeg",
    Content={"S3Bucket": PIPELINE_BUCKET, "S3Key": LAYER_KEY},
    CompatibleRuntimes=["python3.12"], CompatibleArchitectures=["x86_64"])["LayerVersionArn"]
print("Layer:", ffmpeg_layer_arn)

Layer: arn:aws:lambda:us-east-1:346399954218:layer:ffmpeg-static-layer:2


In [23]:
# --- IAM roles for the ffmpeg + transcribe Lambdas (least privilege) ---
trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow",
         "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]}


def ensure_role(name):
    try:
        return iam.create_role(RoleName=name,
            AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
    except iam.exceptions.EntityAlreadyExistsException:
        return iam.get_role(RoleName=name)["Role"]["Arn"]


ffmpeg_role_arn = ensure_role(FFMPEG_ROLE)
transcribe_role_arn = ensure_role(TRANSCRIBE_ROLE)

flg = f"arn:aws:logs:{region}:{account_id}:log-group:/aws/lambda/{FFMPEG_FN}"
iam.put_role_policy(RoleName=FFMPEG_ROLE, PolicyName="inline", PolicyDocument=json.dumps(
    {"Version": "2012-10-17", "Statement": [
        {"Sid": "Logs", "Effect": "Allow",
         "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
         "Resource": [flg, flg + ":*"]},
        {"Sid": "ReadVideo", "Effect": "Allow", "Action": ["s3:GetObject"],
         "Resource": f"arn:aws:s3:::{PIPELINE_BUCKET}/{VIDEO_PREFIX}*"},
        {"Sid": "WriteAudio", "Effect": "Allow", "Action": ["s3:PutObject"],
         "Resource": f"arn:aws:s3:::{PIPELINE_BUCKET}/{AUDIO_PREFIX}*"}]}))

tlg = f"arn:aws:logs:{region}:{account_id}:log-group:/aws/lambda/{TRANSCRIBE_FN}"
iam.put_role_policy(RoleName=TRANSCRIBE_ROLE, PolicyName="inline", PolicyDocument=json.dumps(
    {"Version": "2012-10-17", "Statement": [
        {"Sid": "Logs", "Effect": "Allow",
         "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
         "Resource": [tlg, tlg + ":*"]},
        {"Sid": "InvokeAsync", "Effect": "Allow", "Action": ["sagemaker:InvokeEndpointAsync"],
         "Resource": f"arn:aws:sagemaker:{region}:{account_id}:endpoint/{WORD_ENDPOINT}"}]}))
time.sleep(12)
print("Roles ready")

Roles ready


In [ ]:
# --- create/update the ffmpeg + transcribe Lambdas ---
FFMPEG_SRC = 'import os\nimport urllib.parse\nimport subprocess\nimport boto3\n\ns3 = boto3.client("s3")\nAUDIO_BUCKET = os.environ["AUDIO_BUCKET"]\nAUDIO_PREFIX = os.environ.get("AUDIO_PREFIX", "audio/")\nFFMPEG = "/opt/bin/ffmpeg"\n\n\ndef handler(event, context):\n    for rec in event.get("Records", []):\n        src_bucket = rec["s3"]["bucket"]["name"]\n        key = urllib.parse.unquote_plus(rec["s3"]["object"]["key"])\n        base = os.path.splitext(os.path.basename(key))[0]\n        in_path = "/tmp/" + base + ".src"\n        out_path = "/tmp/" + base + ".wav"\n\n        print("Downloading s3://%s/%s" % (src_bucket, key))\n        s3.download_file(src_bucket, key, in_path)\n\n        cmd = [FFMPEG, "-y", "-i", in_path, "-vn",\n               "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", out_path]\n        proc = subprocess.run(cmd, capture_output=True)\n        if proc.returncode != 0:\n            print(proc.stderr.decode("utf-8", "replace")[-2000:])\n            raise RuntimeError("ffmpeg failed for " + key)\n\n        out_key = AUDIO_PREFIX + base + ".wav"\n        s3.upload_file(out_path, AUDIO_BUCKET, out_key)\n        print("Wrote s3://%s/%s" % (AUDIO_BUCKET, out_key))\n        for p in (in_path, out_path):\n            try:\n                os.remove(p)\n            except OSError:\n                pass\n    return {"status": "ok"}\n'
TRANSCRIBE_SRC = 'import os\nimport urllib.parse\nimport boto3\n\nsmr = boto3.client("sagemaker-runtime")\nENDPOINT_NAME = os.environ["ENDPOINT_NAME"]\n\n\ndef handler(event, context):\n    for rec in event.get("Records", []):\n        bucket = rec["s3"]["bucket"]["name"]\n        key = urllib.parse.unquote_plus(rec["s3"]["object"]["key"])\n        input_uri = "s3://%s/%s" % (bucket, key)\n        resp = smr.invoke_endpoint_async(\n            EndpointName=ENDPOINT_NAME,\n            InputLocation=input_uri,\n            ContentType="audio/wav",\n            Accept="application/json",\n            InvocationTimeoutSeconds=3600,\n        )\n        print("Submitted %s -> %s" % (input_uri, resp.get("OutputLocation")))\n    return {"status": "ok"}\n'


def deploy_fn(name, role_arn, src, env, layers=None, memory=2048, timeout=900, storage=10240):
    code = zip_lambda(src)
    try:
        lam.create_function(FunctionName=name, Runtime="python3.12", Role=role_arn,
            Handler="lambda_function.handler", Code={"ZipFile": code},
            MemorySize=memory, Timeout=timeout, EphemeralStorage={"Size": storage},
            Architectures=["x86_64"], Environment={"Variables": env}, Layers=layers or [])
        print("Created", name)
    except lam.exceptions.ResourceConflictException:
        lam.update_function_code(FunctionName=name, ZipFile=code)
        lam.get_waiter("function_updated_v2").wait(FunctionName=name)
        lam.update_function_configuration(FunctionName=name, Role=role_arn,
            MemorySize=memory, Timeout=timeout, EphemeralStorage={"Size": storage},
            Environment={"Variables": env}, Layers=layers or [])
        print("Updated", name)
    lam.get_waiter("function_active_v2").wait(FunctionName=name)


deploy_fn(FFMPEG_FN, ffmpeg_role_arn, FFMPEG_SRC,
          {"AUDIO_BUCKET": PIPELINE_BUCKET, "AUDIO_PREFIX": AUDIO_PREFIX},
          layers=[ffmpeg_layer_arn], memory=4096)
deploy_fn(TRANSCRIBE_FN, transcribe_role_arn, TRANSCRIBE_SRC,
          {"ENDPOINT_NAME": WORD_ENDPOINT}, memory=256, timeout=60, storage=512)

In [ ]:
# --- wire the ingestion triggers on the pipeline bucket ---
for fn_name, sid in [(FFMPEG_FN, "s3invoke-ffmpeg"), (TRANSCRIBE_FN, "s3invoke-transcribe")]:
    try:
        lam.add_permission(FunctionName=fn_name, StatementId=sid,
            Action="lambda:InvokeFunction", Principal="s3.amazonaws.com",
            SourceArn=f"arn:aws:s3:::{PIPELINE_BUCKET}", SourceAccount=account_id)
    except lam.exceptions.ResourceConflictException:
        pass

ffmpeg_arn = lam.get_function(FunctionName=FFMPEG_FN)["Configuration"]["FunctionArn"]
transcribe_arn = lam.get_function(FunctionName=TRANSCRIBE_FN)["Configuration"]["FunctionArn"]
s3.put_bucket_notification_configuration(Bucket=PIPELINE_BUCKET, NotificationConfiguration={
    "LambdaFunctionConfigurations": [
        {"Id": "video-to-ffmpeg", "LambdaFunctionArn": ffmpeg_arn,
         "Events": ["s3:ObjectCreated:*"],
         "Filter": {"Key": {"FilterRules": [{"Name": "prefix", "Value": VIDEO_PREFIX}]}}},
        {"Id": "wav-to-transcribe", "LambdaFunctionArn": transcribe_arn,
         "Events": ["s3:ObjectCreated:*"],
         "Filter": {"Key": {"FilterRules": [{"Name": "prefix", "Value": AUDIO_PREFIX},
                                            {"Name": "suffix", "Value": ".wav"}]}}}]})
print("Ingestion triggers wired:", VIDEO_PREFIX, "->ffmpeg ->", AUDIO_PREFIX, "->transcribe")

## 3. Translation + subtitles Lambda (Bedrock / Claude)

In [24]:
# --- deploy the translate+subtitles Lambda ---
try:
    translate_role_arn = iam.create_role(RoleName=TRANSLATE_ROLE,
        AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
except iam.exceptions.EntityAlreadyExistsException:
    translate_role_arn = iam.get_role(RoleName=TRANSLATE_ROLE)["Role"]["Arn"]

xlg = f"arn:aws:logs:{region}:{account_id}:log-group:/aws/lambda/{TRANSLATE_FN}"
iam.put_role_policy(RoleName=TRANSLATE_ROLE, PolicyName="inline", PolicyDocument=json.dumps(
    {"Version": "2012-10-17", "Statement": [
        {"Sid": "Logs", "Effect": "Allow",
         "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
         "Resource": [xlg, xlg + ":*"]},
        {"Sid": "ReadWords", "Effect": "Allow", "Action": ["s3:GetObject"],
         "Resource": f"arn:aws:s3:::{bucket}/{WORD_OUTPUT_PREFIX}*"},
        {"Sid": "WriteSubs", "Effect": "Allow", "Action": ["s3:PutObject"],
         "Resource": f"arn:aws:s3:::{PIPELINE_BUCKET}/{EN_PREFIX}*"},
        {"Sid": "InvokeBedrock", "Effect": "Allow", "Action": ["bedrock:InvokeModel"],
         "Resource": [f"arn:aws:bedrock:{region}:{account_id}:inference-profile/{MODEL_ID}",
                      "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-opus-4-8",
                      "arn:aws:bedrock:us-east-2::foundation-model/anthropic.claude-opus-4-8",
                      "arn:aws:bedrock:us-west-2::foundation-model/anthropic.claude-opus-4-8"]}]}))
time.sleep(12)

TRANSLATE_SRC = 'import os\nimport re\nimport json\nimport urllib.parse\nimport boto3\n\ns3 = boto3.client("s3")\nbedrock = boto3.client("bedrock-runtime",\n                       region_name=os.environ.get("BEDROCK_REGION", "us-east-1"))\n\nMODEL_ID = os.environ.get("MODEL_ID", "us.anthropic.claude-opus-4-8")\nOUT_BUCKET = os.environ["OUT_BUCKET"]\nOUT_PREFIX = os.environ.get("OUT_PREFIX", "transcripts-en/")\nMAX_DUR = float(os.environ.get("MAX_CUE_SECONDS", "6"))\nMAX_CHARS = int(os.environ.get("MAX_CUE_CHARS", "84"))\nPAUSE_GAP = float(os.environ.get("PAUSE_GAP", "0.7"))\nBATCH = int(os.environ.get("TRANSLATE_BATCH", "30"))\nLINE_LEN = int(os.environ.get("SRT_LINE_LEN", "42"))\n\nSENTENCE_END = set(".?!\\u2026:;")\n\nSUBTITLE_SYSTEM = (\n    "You are an expert Yiddish-to-English subtitle translator. You receive consecutive "\n    "subtitle lines (Yiddish in Hebrew script) from automatically transcribed speech. "\n    "Use the surrounding lines as context to translate each line into clear, natural, "\n    "idiomatic English suitable as a subtitle. Translate faithfully without adding, "\n    "omitting, summarizing, or explaining. Render religious, Talmudic, and cultural "\n    "terms accurately; keep names in standard English spelling; leave already-English "\n    "text unchanged. Return ONLY a JSON array of strings: exactly one translation per "\n    "input line, in the same order and the same count."\n)\n\n\ndef parse_transcript(data):\n    candidates = data if isinstance(data, list) else [data]\n    for c in candidates:\n        if isinstance(c, str):\n            try:\n                c = json.loads(c)\n            except (ValueError, TypeError):\n                continue\n        if isinstance(c, dict) and ("chunks" in c or "text" in c):\n            return c\n    return {"text": "", "chunks": []}\n\n\ndef mk_cue(words):\n    text = "".join(w["text"] for w in words).strip()\n    return {"start": words[0]["timestamp"][0],\n            "end": words[-1]["timestamp"][1], "he": text}\n\n\ndef build_sentence_cues(words):\n    cues, cur = [], []\n    for w in words:\n        ts = w.get("timestamp") or [None, None]\n        ws, we = ts[0], ts[1]\n        if w.get("text") is None or ws is None or we is None:\n            continue\n        if cur:\n            cstart = cur[0]["timestamp"][0]\n            cend = cur[-1]["timestamp"][1]\n            ctext = "".join(x["text"] for x in cur)\n            gap = ws - cend if cend is not None else 0.0\n            dur = we - cstart if cstart is not None else 0.0\n            if gap >= PAUSE_GAP or dur >= MAX_DUR or len(ctext) >= MAX_CHARS:\n                cues.append(mk_cue(cur))\n                cur = []\n        cur.append(w)\n        wt = w["text"].strip()\n        if wt and wt[-1] in SENTENCE_END:\n            cues.append(mk_cue(cur))\n            cur = []\n    if cur:\n        cues.append(mk_cue(cur))\n    return cues\n\n\ndef extract_json_array(s):\n    s = s.strip()\n    s = re.sub(r"^```(?:json)?|```$", "", s, flags=re.MULTILINE).strip()\n    i, j = s.find("["), s.rfind("]")\n    if i != -1 and j != -1 and j > i:\n        return json.loads(s[i:j + 1])\n    raise ValueError("no JSON array")\n\n\ndef translate_one(text):\n    text = (text or "").strip()\n    if not text:\n        return ""\n    r = bedrock.converse(\n        modelId=MODEL_ID, system=[{"text": SUBTITLE_SYSTEM}],\n        messages=[{"role": "user",\n                   "content": [{"text": json.dumps([text], ensure_ascii=False)}]}],\n        inferenceConfig={"maxTokens": 1024},\n    )\n    out = r["output"]["message"]["content"][0]["text"]\n    try:\n        return extract_json_array(out)[0]\n    except Exception:\n        return out.strip()\n\n\ndef translate_cues(cues):\n    texts = [c["he"] for c in cues]\n    results = []\n    for i in range(0, len(texts), BATCH):\n        seg = texts[i:i + BATCH]\n        user = ("Translate these consecutive Yiddish subtitle lines to English. "\n                "Return a JSON array of the same length and order.\\n"\n                + json.dumps(seg, ensure_ascii=False))\n        r = bedrock.converse(\n            modelId=MODEL_ID, system=[{"text": SUBTITLE_SYSTEM}],\n            messages=[{"role": "user", "content": [{"text": user}]}],\n            inferenceConfig={"maxTokens": 4096},\n        )\n        out = r["output"]["message"]["content"][0]["text"]\n        try:\n            arr = extract_json_array(out)\n            if len(arr) != len(seg):\n                raise ValueError("count mismatch")\n        except Exception:\n            arr = [translate_one(t) for t in seg]\n        results.extend(arr)\n    for c, t in zip(cues, results):\n        c["en"] = t\n    return cues\n\n\ndef fmt_ts(sec):\n    sec = max(0.0, sec or 0.0)\n    ms = int(round(sec * 1000))\n    h, ms = divmod(ms, 3600000)\n    m, ms = divmod(ms, 60000)\n    s, ms = divmod(ms, 1000)\n    return "%02d:%02d:%02d,%03d" % (h, m, s, ms)\n\n\ndef wrap(text):\n    words = text.split()\n    lines, cur = [], ""\n    for w in words:\n        if cur and len(cur) + 1 + len(w) > LINE_LEN:\n            lines.append(cur)\n            cur = w\n        else:\n            cur = (cur + " " + w).strip()\n    if cur:\n        lines.append(cur)\n    if len(lines) > 2:\n        lines = [lines[0], " ".join(lines[1:])]\n    return "\\n".join(lines)\n\n\ndef to_srt(cues):\n    out = []\n    for i, c in enumerate(cues, 1):\n        start = c["start"]\n        end = max(c["end"], start + 1.0)\n        body = wrap((c.get("en") or c["he"]).strip())\n        out.append("%d\\n%s --> %s\\n%s\\n" % (i, fmt_ts(start), fmt_ts(end), body))\n    return "\\n".join(out)\n\n\ndef handler(event, context):\n    for rec in event.get("Records", []):\n        bucket = rec["s3"]["bucket"]["name"]\n        key = urllib.parse.unquote_plus(rec["s3"]["object"]["key"])\n        print("Reading words s3://%s/%s" % (bucket, key))\n        raw = s3.get_object(Bucket=bucket, Key=key)["Body"].read()\n        words = parse_transcript(json.loads(raw)).get("chunks", [])\n        cues = translate_cues(build_sentence_cues(words))\n\n        base = os.path.splitext(os.path.basename(key))[0]\n        doc = {"text": " ".join(c.get("en", "") for c in cues), "cues": cues}\n        s3.put_object(Bucket=OUT_BUCKET, Key=OUT_PREFIX + base + ".en.json",\n                      Body=json.dumps(doc, ensure_ascii=False).encode("utf-8"),\n                      ContentType="application/json")\n        s3.put_object(Bucket=OUT_BUCKET, Key=OUT_PREFIX + base + ".en.srt",\n                      Body=to_srt(cues).encode("utf-8"),\n                      ContentType="text/plain; charset=utf-8")\n        print("Wrote %s%s.en.json and .en.srt" % (OUT_PREFIX, base))\n    return {"status": "ok"}\n'

env = {"MODEL_ID": MODEL_ID, "BEDROCK_REGION": BEDROCK_REGION,
       "OUT_BUCKET": PIPELINE_BUCKET, "OUT_PREFIX": EN_PREFIX,
       "MAX_CUE_SECONDS": "6", "MAX_CUE_CHARS": "84", "PAUSE_GAP": "0.7",
       "TRANSLATE_BATCH": "30", "SRT_LINE_LEN": "42"}
code = zip_lambda(TRANSLATE_SRC)
try:
    lam.create_function(FunctionName=TRANSLATE_FN, Runtime="python3.12",
        Role=translate_role_arn, Handler="lambda_function.handler",
        Code={"ZipFile": code}, MemorySize=512, Timeout=900,
        Architectures=["x86_64"], Environment={"Variables": env})
    print("Created", TRANSLATE_FN)
except lam.exceptions.ResourceConflictException:
    lam.update_function_code(FunctionName=TRANSLATE_FN, ZipFile=code)
    lam.get_waiter("function_updated_v2").wait(FunctionName=TRANSLATE_FN)
    lam.update_function_configuration(FunctionName=TRANSLATE_FN, Role=translate_role_arn,
        MemorySize=512, Timeout=900, Environment={"Variables": env})
    print("Updated", TRANSLATE_FN)
lam.get_waiter("function_active_v2").wait(FunctionName=TRANSLATE_FN)

Created whisper-translate-subtitles


In [25]:
# --- wire the translate trigger on the word-output prefix (non-destructive merge) ---
try:
    lam.add_permission(FunctionName=TRANSLATE_FN, StatementId="s3invoke-translate",
        Action="lambda:InvokeFunction", Principal="s3.amazonaws.com",
        SourceArn=f"arn:aws:s3:::{bucket}", SourceAccount=account_id)
except lam.exceptions.ResourceConflictException:
    pass

translate_arn = lam.get_function(FunctionName=TRANSLATE_FN)["Configuration"]["FunctionArn"]
existing = s3.get_bucket_notification_configuration(Bucket=bucket)
clean = {k: existing[k] for k in ("TopicConfigurations", "QueueConfigurations",
         "LambdaFunctionConfigurations", "EventBridgeConfiguration") if k in existing}
# drop any of our previous translate triggers, then add the word-output one
keep = [c for c in clean.get("LambdaFunctionConfigurations", [])
        if c.get("Id") not in ("transcript-to-translate", "wordoutput-to-translate")]
keep.append({"Id": "wordoutput-to-translate", "LambdaFunctionArn": translate_arn,
             "Events": ["s3:ObjectCreated:*"],
             "Filter": {"Key": {"FilterRules": [
                 {"Name": "prefix", "Value": WORD_OUTPUT_PREFIX},
                 {"Name": "suffix", "Value": ".out"}]}}})
clean["LambdaFunctionConfigurations"] = keep
s3.put_bucket_notification_configuration(Bucket=bucket, NotificationConfiguration=clean)
print("Translate trigger wired on", f"s3://{bucket}/{WORD_OUTPUT_PREFIX}")

Translate trigger wired on s3://sagemaker-us-east-1-346399954218/whisper/word-output/


## 4. Run end-to-end

In [10]:
# --- upload a video and follow it to finished subtitles (progress + diagnostics) ---
logs = boto3.client("logs")
LOCAL_VIDEO = "1.mp4"   # <-- set to your local video file


def list_objs(b, p):
    """Return {key: LastModified} so overwrites of an existing key are detected too."""
    out, token = {}, None
    while True:
        kw = {"Bucket": b, "Prefix": p}
        if token:
            kw["ContinuationToken"] = token
        r = s3.list_objects_v2(**kw)
        for o in r.get("Contents", []):
            out[o["Key"]] = o["LastModified"]
        if r.get("IsTruncated"):
            token = r["NextContinuationToken"]
        else:
            return out


def newer_keys(current, baseline):
    return sorted(k for k, lm in current.items() if baseline.get(k) != lm)


def tail_lambda_logs(fn_name, minutes=20, limit=40):
    """Print the most recent CloudWatch log lines for a Lambda, to diagnose failures."""
    grp = f"/aws/lambda/{fn_name}"
    print(f"   CloudWatch log group: {grp}")
    try:
        streams = logs.describe_log_streams(
            logGroupName=grp, orderBy="LastEventTime", descending=True, limit=1
        ).get("logStreams", [])
    except logs.exceptions.ResourceNotFoundException:
        print(f"   (no log group yet - {fn_name} may not have been invoked)")
        return
    if not streams:
        print("   (no log streams yet)")
        return
    start = int((time.time() - minutes * 60) * 1000)
    ev = logs.get_log_events(logGroupName=grp, logStreamName=streams[0]["logStreamName"],
                             startTime=start, limit=limit, startFromHead=False).get("events", [])
    if not ev:
        print(f"   (no log lines in the last {minutes} min)")
        return
    print(f"   --- last {len(ev)} log lines for {fn_name} ---")
    for e in ev:
        print("   |", e["message"].rstrip())


def show_failures(b, prefix, baseline):
    new = newer_keys(list_objs(b, prefix), baseline)
    for k in new:
        body = s3.get_object(Bucket=b, Key=k)["Body"].read().decode("utf-8", "replace")
        print(f"   failure object s3://{b}/{k}:")
        print("   |", body[:600])
    return bool(new)


def wait_stage(b, p, baseline, label, fn_for_logs, timeout=1800, poll=10,
               fail_bucket=None, fail_prefix=None, fail_baseline=None, extra_hint=None):
    print(f"\n[stage] {label} - waiting (timeout {timeout}s) ...")
    deadline = time.time() + timeout
    while time.time() < deadline:
        new = newer_keys(list_objs(b, p), baseline)
        if new:
            return new[0]
        if fail_prefix and show_failures(fail_bucket, fail_prefix, fail_baseline or {}):
            print(f"  [FAILED] {label}: an error was written to {fail_prefix}")
            if extra_hint:
                print("  hint:", extra_hint)
            print(f"  -> Check the CloudWatch logs for {fn_for_logs}:")
            tail_lambda_logs(fn_for_logs)
            return None
        time.sleep(poll)
    print(f"  [TIMEOUT] {label} did not complete within {timeout}s.")
    if extra_hint:
        print("  hint:", extra_hint)
    print(f"  -> Check the CloudWatch logs for {fn_for_logs}:")
    tail_lambda_logs(fn_for_logs)
    return None


def run_pipeline(local_video):
    base_audio = list_objs(PIPELINE_BUCKET, AUDIO_PREFIX)
    base_words = list_objs(bucket, WORD_OUTPUT_PREFIX)
    base_subs = list_objs(PIPELINE_BUCKET, EN_PREFIX)
    base_wordfail = list_objs(bucket, "whisper/word-failure/")

    vid_key = VIDEO_PREFIX + os.path.basename(local_video)
    s3.upload_file(local_video, PIPELINE_BUCKET, vid_key)
    print(f">>> Uploaded video: s3://{PIPELINE_BUCKET}/{vid_key}")

    k = wait_stage(PIPELINE_BUCKET, AUDIO_PREFIX, base_audio,
                   "Audio extraction (ffmpeg)", FFMPEG_FN, timeout=900,
                   extra_hint="ffmpeg failed to decode the video, or the layer/binary is missing.")
    if not k:
        print("\nStopped at the AUDIO stage.")
        return
    print(f">>> Audio extracted and stored: s3://{PIPELINE_BUCKET}/{k}")

    k = wait_stage(bucket, WORD_OUTPUT_PREFIX, base_words,
                   "Transcription (whisper-yi-word)", TRANSCRIBE_FN, timeout=1800,
                   fail_bucket=bucket, fail_prefix="whisper/word-failure/",
                   fail_baseline=base_wordfail,
                   extra_hint=(f"If the transcribe Lambda logs look clean, the endpoint crashed - "
                               f"check /aws/sagemaker/Endpoints/{WORD_ENDPOINT} (a 'Worker died' 500 "
                               f"usually means GPU OOM; lower batch_size)."))
    if not k:
        print("\nStopped at the TRANSCRIPTION stage.")
        return
    print(f">>> Transcription generated and stored: s3://{bucket}/{k}")

    k = wait_stage(PIPELINE_BUCKET, EN_PREFIX, base_subs,
                   "Translation + subtitles (Bedrock)", TRANSLATE_FN, timeout=1800,
                   extra_hint=("Common causes: Bedrock model access not enabled, wrong MODEL_ID, "
                               "or an inferenceConfig param the model rejects."))
    if not k:
        print("\nStopped at the TRANSLATION stage.")
        return
    print(f">>> Subtitles generated and stored: s3://{PIPELINE_BUCKET}/{k}")
    print("\n=== DONE === Run the next cell to download the .srt.")


run_pipeline(LOCAL_VIDEO)

>>> Uploaded video: s3://whisper-video-pipeline-346399954218-us-east-1/videos/1.mp4

[stage] Audio extraction (ffmpeg) - waiting (timeout 900s) ...
>>> Audio extracted and stored: s3://whisper-video-pipeline-346399954218-us-east-1/audio/1.wav

[stage] Transcription (whisper-yi-word) - waiting (timeout 1800s) ...
   failure object s3://sagemaker-us-east-1-346399954218/whisper/word-failure/dc2001de-6b38-4a0a-bb01-f67595febaff-error.out:
   | {
  "code": 500,
  "type": "InternalServerException",
  "message": "Worker died."
}

  [FAILED] Transcription (whisper-yi-word): an error was written to whisper/word-failure/
  hint: If the transcribe Lambda logs look clean, the endpoint crashed - check /aws/sagemaker/Endpoints/whisper-yi-word (a 'Worker died' 500 usually means GPU OOM; lower batch_size).
  -> Check the CloudWatch logs for whisper-invoke-transcribe:
   CloudWatch log group: /aws/lambda/whisper-invoke-transcribe
   --- last 5 log lines for whisper-invoke-transcribe ---
   | INIT_START

In [ ]:
# --- fetch the latest subtitles ---
objs = [o for o in s3.list_objects_v2(Bucket=PIPELINE_BUCKET, Prefix=EN_PREFIX).get("Contents", [])
        if o["Key"].endswith(".en.srt")]
if not objs:
    raise FileNotFoundError("no .en.srt yet under " + EN_PREFIX)
key = sorted(objs, key=lambda o: o["LastModified"])[-1]["Key"]
local = os.path.basename(key)
s3.download_file(PIPELINE_BUCKET, key, local)
print("Downloaded", local, "\n")
print("\n".join(open(local, encoding="utf-8").read().split("\n\n")[:6]))

## Decommission the old chunk-level endpoint (option 2)

In [28]:
# --- delete whisper-yi-model (no longer needed; word-level supersedes it) ---
OLD_ENDPOINT = "whisper-yi-word"
for fn, kw in [(sm.delete_endpoint, {"EndpointName": OLD_ENDPOINT}),
               (sm.delete_endpoint_config, {"EndpointConfigName": OLD_ENDPOINT}),
               (sm.delete_model, {"ModelName": OLD_ENDPOINT})]:
    try:
        fn(**kw); print("Deleted", kw)
    except Exception as e:
        print("Skip", kw, "->", type(e).__name__)

Deleted {'EndpointName': 'whisper-yi-word'}
Deleted {'EndpointConfigName': 'whisper-yi-word'}
Deleted {'ModelName': 'whisper-yi-word'}


## Teardown (optional)

Uncomment to remove everything this notebook created.

In [ ]:
# # detach the translate trigger
# existing = s3.get_bucket_notification_configuration(Bucket=bucket)
# clean = {k: existing[k] for k in ("TopicConfigurations", "QueueConfigurations",
#          "LambdaFunctionConfigurations", "EventBridgeConfiguration") if k in existing}
# clean["LambdaFunctionConfigurations"] = [c for c in clean.get("LambdaFunctionConfigurations", [])
#     if c.get("Id") != "wordoutput-to-translate"]
# s3.put_bucket_notification_configuration(Bucket=bucket, NotificationConfiguration=clean)
# # lambdas
# for fn in (FFMPEG_FN, TRANSCRIBE_FN, TRANSLATE_FN):
#     try: lam.delete_function(FunctionName=fn)
#     except lam.exceptions.ResourceNotFoundException: pass
# # layer versions
# for v in lam.list_layer_versions(LayerName=FFMPEG_LAYER).get("LayerVersions", []):
#     lam.delete_layer_version(LayerName=FFMPEG_LAYER, VersionNumber=v["Version"])
# # roles
# for r, in [(FFMPEG_ROLE,), (TRANSCRIBE_ROLE,), (TRANSLATE_ROLE,)]:
#     try: iam.delete_role_policy(RoleName=r, PolicyName="inline")
#     except iam.exceptions.NoSuchEntityException: pass
#     try: iam.delete_role(RoleName=r)
#     except iam.exceptions.NoSuchEntityException: pass
# # endpoint
# for fn, kw in [(sm.delete_endpoint, {"EndpointName": WORD_ENDPOINT}),
#                (sm.delete_endpoint_config, {"EndpointConfigName": WORD_ENDPOINT}),
#                (sm.delete_model, {"ModelName": WORD_ENDPOINT})]:
#     try: fn(**kw)
#     except Exception: pass
# # empty + delete bucket
# for page in s3.get_paginator("list_objects_v2").paginate(Bucket=PIPELINE_BUCKET):
#     for o in page.get("Contents", []):
#         s3.delete_object(Bucket=PIPELINE_BUCKET, Key=o["Key"])
# s3.delete_bucket(Bucket=PIPELINE_BUCKET)
# print("Torn down")
# # dedicated endpoint role
# try: iam.delete_role_policy(RoleName=ENDPOINT_ROLE, PolicyName="inline")
# except iam.exceptions.NoSuchEntityException: pass
# try: iam.delete_role(RoleName=ENDPOINT_ROLE)
# except iam.exceptions.NoSuchEntityException: pass